# Kaggle probe — schema-native decision trackRun top to bottom. **Notebook options (right sidebar): Internet = On, Accelerator = None.**No GPU is needed for this notebook; turn it on only when a runner actually loads a model.The point of this notebook is to answer three questions that cannot be answered fromoutside Kaggle, and that block writing `src/runners/*`:1. what environment do we actually get,2. what is the real Laya package/API and parameter count,3. do the ladder's model IDs exist, and under what licence.Paste the whole output back into the session.

## 1. Environment

In [ ]:
import platform, sysprint("python", platform.python_version())print("executable", sys.executable)!nvidia-smi -L || echo "no GPU attached (fine for this notebook)"!pip list 2>/dev/null | grep -iE "^(torch|transformers|tokenizers|accelerate|outlines|xgrammar|huggingface-hub|pydantic) " 

## 2. The harness itself

Clones the repo into the writable working directory. Needs **Internet = On** in the
sidebar. The repo is public, so no token is involved.

(If it is private again later: put a GitHub token in Add-ons -> Secrets as `GITHUB_TOKEN`,
read it with `UserSecretsClient().get_secret("GITHUB_TOKEN")`, and clone
`https://$token@github.com/...`. Never paste a token into a cell.)

In [ ]:
# Step out of the target directory FIRST: deleting the directory the
# kernel is standing in breaks getcwd for every later shell command.
%cd /kaggle/working
!rm -rf /kaggle/working/HARNESS
!git clone --depth 1 https://github.com/Vansh-kap-98/HARNESS.git /kaggle/working/HARNESS
%cd /kaggle/working/HARNESS
!ls -R src tests schemas | head -40

In [ ]:
!pip install -q pytest jsonschema
!cd /kaggle/working/HARNESS && python -m pytest -q

In [ ]:
!cd /kaggle/working/HARNESS && python -m src.compiler schemas/support_ticket.json
!cd /kaggle/working/HARNESS && python -m src.dataset tests/fixtures/mini.jsonl schemas/support_ticket.json

## 3. Laya: the real API surfaceThis is the blocking one. The model card text is what the runner has to be written against --no guessing.

In [ ]:
from huggingface_hub import model_info, hf_hub_downloadREPO = "convaiinnovations/laya"info = model_info(REPO)print("id:", info.id)print("pipeline_tag:", info.pipeline_tag)print("tags:", info.tags)print("downloads:", info.downloads, " likes:", info.likes)print("files:")for sibling in info.siblings:    print("  ", sibling.rfilename)

In [ ]:
card = hf_hub_download(REPO, "README.md")text = open(card, encoding="utf-8").read()print(len(text), "characters of model card")print(text[:12000])

In [ ]:
import json

# The encoder configs are under encoder/, not at the repo root.
for filename in ["encoder/config.json", "multilingual/encoder/config.json",
                 "typed-decisions/encoder/config.json", "rl_agent_config.json",
                 "eval/results.json"]:
    try:
        path = hf_hub_download(REPO, filename)
        print("=== " + filename + " ===")
        print(json.dumps(json.load(open(path, encoding="utf-8")), indent=2)[:2000])
    except Exception as exc:
        print("=== " + filename + " === not available: " + repr(exc))

### Only if the card names a pip packageReplace `PACKAGE` with whatever the card says to install. If it says to use `transformers`directly, skip this cell and say so.

In [ ]:
# !pip install -q PACKAGE# import PACKAGE as laya, inspect# print(getattr(laya, "__version__", "no __version__"))# print([n for n in dir(laya) if not n.startswith("_")])# print(inspect.signature(laya.load))

## 4. The rest of the ladder: do these IDs exist, and under what licence?

In [ ]:
from huggingface_hub import model_infoLADDER = [    "Qwen/Qwen2.5-0.5B-Instruct",    "Qwen/Qwen2.5-1.5B-Instruct",    "Qwen/Qwen2.5-3B-Instruct",    "Qwen/Qwen2.5-7B-Instruct",]for repo in LADDER:    try:        info = model_info(repo)        licences = [t for t in info.tags if t.startswith("license:")]        print("%-32s OK  %s" % (repo, licences or "no license tag"))    except Exception as exc:        print("%-32s FAILED  %s" % (repo, type(exc).__name__))

## 5. Constrained decoding: which library, which versionWhatever is preinstalled decides which API the baseline runner is written against.

In [ ]:
for name in ["outlines", "xgrammar"]:    try:        module = __import__(name)        print(name, getattr(module, "__version__", "installed, no __version__"))        print("  top level:", [n for n in dir(module) if not n.startswith("_")][:25])    except ImportError:        print(name, "NOT installed")